Patient Education Chatbot with RAG and Safety Guardrails

This notebook builds a beginner-friendly patient education chatbot.

The chatbot answers general health questions using a small trusted medical Q&A dataset.

The system uses:

a synthetic medical education dataset
retrieval-based search
an open-source language model
emergency keyword detection
unsafe response filtering
patient-friendly explanations

This project is for education and portfolio use only.

It does not diagnose, prescribe, or replace a healthcare professional.

Project Workflow

The workflow is:

Patient question
↓
Safety check for emergency keywords
↓
Search the medical Q&A dataset
↓
Retrieve the most relevant trusted context
↓
Send the context to a language model
↓
Generate a patient-friendly answer
↓
Check the output for unsafe medical claims
↓
Show final answer with disclaimer

This is a simple RAG system.

RAG means Retrieval-Augmented Generation.

In simple words:

The chatbot first searches a trusted dataset, then uses that information to generate an answer.

In [1]:
!pip install pandas numpy scikit-learn transformers torch accelerate sentencepiece

Step 1: Import Libraries

We need:

pandas for tables
numpy for calculations
scikit-learn for similarity search
transformers for the language model
torch to run the model
re for text cleaning and keyword checks

In [1]:
import pandas as pd
import numpy as np
import re
import torch
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
print("GPU available:", torch.cuda.is_available())

Libraries imported successfully.
GPU available: True


Step 2: Create a Synthetic Patient Education Dataset

This dataset contains trusted educational Q&A examples.

Each row has:

category
question
trusted_answer

Important:

This is synthetic educational content for learning.

In a real healthcare chatbot, this dataset should come from trusted clinical sources and be reviewed by medical professionals.

In [2]:
medical_qa_data = [
    {
        "category": "Cardiology",
        "question": "What is high blood pressure and why is it dangerous?",
        "trusted_answer": (
            "High blood pressure means the force of blood against artery walls is consistently too high. "
            "Over time, it can increase the risk of heart disease, stroke, kidney disease, and other health problems. "
            "Lifestyle changes and medications may help, but patients should follow advice from a healthcare professional."
        )
    },
    {
        "category": "Cardiology",
        "question": "What are common symptoms of a heart attack?",
        "trusted_answer": (
            "Possible heart attack symptoms include chest pain or pressure, shortness of breath, sweating, nausea, "
            "pain in the arm, jaw, neck, or back, and unusual fatigue. These symptoms can be an emergency and need urgent care."
        )
    },
    {
        "category": "Diabetes",
        "question": "What are early signs of diabetes?",
        "trusted_answer": (
            "Early signs of diabetes may include increased thirst, frequent urination, fatigue, blurry vision, slow-healing wounds, "
            "and unexplained weight changes. A blood test is needed to confirm diabetes."
        )
    },
    {
        "category": "Diabetes",
        "question": "What is HbA1c?",
        "trusted_answer": (
            "HbA1c is a blood test that shows average blood sugar levels over about the past 2 to 3 months. "
            "It is often used to monitor diabetes control."
        )
    },
    {
        "category": "Nutrition",
        "question": "How can I lower cholesterol naturally?",
        "trusted_answer": (
            "Healthy steps may include eating more fiber-rich foods, choosing unsaturated fats, reducing trans fats, "
            "exercising regularly, avoiding smoking, and maintaining a healthy weight. Some people still need medication."
        )
    },
    {
        "category": "Nutrition",
        "question": "What foods are good for heart health?",
        "trusted_answer": (
            "Heart-healthy foods include vegetables, fruits, whole grains, beans, nuts, seeds, fish, and healthy oils. "
            "Limiting highly processed foods, excess salt, and sugary drinks can also help."
        )
    },
    {
        "category": "Mental Health",
        "question": "What are common signs of anxiety?",
        "trusted_answer": (
            "Common signs of anxiety can include excessive worry, restlessness, trouble sleeping, fast heartbeat, muscle tension, "
            "and difficulty concentrating. Persistent or severe symptoms should be discussed with a healthcare professional."
        )
    },
    {
        "category": "Mental Health",
        "question": "How can I manage stress?",
        "trusted_answer": (
            "Stress management may include regular physical activity, sleep routines, relaxation breathing, talking with supportive people, "
            "time management, and seeking professional help when stress becomes overwhelming."
        )
    },
    {
        "category": "General Wellness",
        "question": "How can I improve my sleep quality?",
        "trusted_answer": (
            "Good sleep habits include keeping a regular sleep schedule, reducing screen time before bed, limiting caffeine late in the day, "
            "creating a dark and quiet sleep environment, and relaxing before bedtime."
        )
    },
    {
        "category": "General Wellness",
        "question": "How much exercise should adults get?",
        "trusted_answer": (
            "Many adults benefit from regular aerobic activity and muscle-strengthening exercises. The right amount depends on health status, "
            "age, and fitness level. People with medical conditions should ask a healthcare professional before starting a new plan."
        )
    },
    {
        "category": "Respiratory Health",
        "question": "What are symptoms of pneumonia?",
        "trusted_answer": (
            "Pneumonia symptoms may include cough, fever, chills, chest discomfort, shortness of breath, fatigue, and sometimes confusion in older adults. "
            "Severe symptoms require urgent medical evaluation."
        )
    },
    {
        "category": "Respiratory Health",
        "question": "What is asthma?",
        "trusted_answer": (
            "Asthma is a long-term condition where airways can become inflamed and narrowed, causing wheezing, coughing, chest tightness, "
            "and shortness of breath. Treatment plans should be guided by a healthcare professional."
        )
    },
    {
        "category": "Medication Safety",
        "question": "Why is it important to tell doctors about all my medications?",
        "trusted_answer": (
            "Doctors and pharmacists need to know all medications, supplements, and allergies to reduce the risk of drug interactions, duplicate therapy, "
            "side effects, and unsafe medication choices."
        )
    },
    {
        "category": "Medication Safety",
        "question": "What should I do if I miss a medication dose?",
        "trusted_answer": (
            "What to do after a missed dose depends on the medication. Patients should check the medication instructions or contact a pharmacist or clinician. "
            "They should not double doses unless specifically instructed."
        )
    },
    {
        "category": "Emergency",
        "question": "When should I seek emergency medical care?",
        "trusted_answer": (
            "Emergency care may be needed for symptoms such as severe chest pain, trouble breathing, signs of stroke, severe bleeding, loss of consciousness, "
            "severe allergic reaction, overdose, or thoughts of self-harm."
        )
    }
]

medical_df = pd.DataFrame(medical_qa_data)

medical_df

,category,question,trusted_answer
0,Cardiology,What is high blood pressure and why is it dang...,High blood pressure means the force of blood a...
1,Cardiology,What are common symptoms of a heart attack?,Possible heart attack symptoms include chest p...
2,Diabetes,What are early signs of diabetes?,Early signs of diabetes may include increased ...
3,Diabetes,What is HbA1c?,HbA1c is a blood test that shows average blood...
4,Nutrition,How can I lower cholesterol naturally?,Healthy steps may include eating more fiber-ri...
5,Nutrition,What foods are good for heart health?,"Heart-healthy foods include vegetables, fruits..."
6,Mental Health,What are common signs of anxiety?,Common signs of anxiety can include excessive ...
7,Mental Health,How can I manage stress?,Stress management may include regular physical...
8,General Wellness,How can I improve my sleep quality?,Good sleep habits include keeping a regular sl...
9,General Wellness,How much exercise should adults get?,Many adults benefit from regular aerobic activ...


Step 3: Explore the Dataset

Before building the chatbot, we check:

how many Q&A records we have
what categories exist
sample questions

This is a normal data analysis habit.

Pattern:

Load data
↓
Inspect rows
↓
Check categories
↓
Understand the dataset

In [3]:
print("Total Q&A records:", len(medical_df))
print("Categories:", medical_df["category"].unique())

medical_df["category"].value_counts()

Total Q&A records: 15
Categories: ['Cardiology' 'Diabetes' 'Nutrition' 'Mental Health' 'General Wellness'
 'Respiratory Health' 'Medication Safety' 'Emergency']


,count
category,
Cardiology,2
Diabetes,2
Nutrition,2
Mental Health,2
General Wellness,2
Respiratory Health,2
Medication Safety,2
Emergency,1


In [4]:
for category in medical_df["category"].unique():
    print("\nCategory:", category)
    sample_questions = medical_df[medical_df["category"] == category]["question"].head(2)

    for question in sample_questions:
        print("-", question)


Category: Cardiology
- What is high blood pressure and why is it dangerous?
- What are common symptoms of a heart attack?

Category: Diabetes
- What are early signs of diabetes?
- What is HbA1c?

Category: Nutrition
- How can I lower cholesterol naturally?
- What foods are good for heart health?

Category: Mental Health
- What are common signs of anxiety?
- How can I manage stress?

Category: General Wellness
- How can I improve my sleep quality?
- How much exercise should adults get?

Category: Respiratory Health
- What are symptoms of pneumonia?
- What is asthma?

Category: Medication Safety
- Why is it important to tell doctors about all my medications?
- What should I do if I miss a medication dose?

Category: Emergency
- When should I seek emergency medical care?


Step 4: Build the Retrieval System

This is the RAG part.

We combine the category, question, and trusted answer into one searchable text.

Then we use TF-IDF.

TF-IDF helps compare the user question with the dataset.

Beginner idea:

User question
↓
Convert question into numbers
↓
Compare with dataset rows
↓
Find most similar Q&A examples

In [5]:
medical_df["search_text"] = (
    medical_df["category"] + " " +
    medical_df["question"] + " " +
    medical_df["trusted_answer"]
)

vectorizer = TfidfVectorizer(stop_words="english")

qa_matrix = vectorizer.fit_transform(medical_df["search_text"])

print("Retrieval system created.")
print("Matrix shape:", qa_matrix.shape)

Retrieval system created.
Matrix shape: (15, 242)


Step 5: Create a Retrieve Function

This function searches the dataset and returns the most relevant context.

Input:

User question

Output:

Most relevant trusted Q&A records

Code pattern:

question
↓
vectorize question
↓
calculate similarity
↓
sort by similarity
↓
return top matches

In [6]:
def retrieve_context(user_question, top_k=2):
    question_vector = vectorizer.transform([user_question])

    similarity_scores = cosine_similarity(question_vector, qa_matrix).flatten()

    top_indices = similarity_scores.argsort()[::-1][:top_k]

    retrieved_items = []

    for index in top_indices:
        retrieved_items.append({
            "category": medical_df.iloc[index]["category"],
            "question": medical_df.iloc[index]["question"],
            "trusted_answer": medical_df.iloc[index]["trusted_answer"],
            "similarity_score": round(float(similarity_scores[index]), 3)
        })

    return retrieved_items

In [7]:
test_question = "What is high blood pressure?"

retrieved = retrieve_context(test_question)

retrieved

[{'category': 'Cardiology',
  'question': 'What is high blood pressure and why is it dangerous?',
  'trusted_answer': 'High blood pressure means the force of blood against artery walls is consistently too high. Over time, it can increase the risk of heart disease, stroke, kidney disease, and other health problems. Lifestyle changes and medications may help, but patients should follow advice from a healthcare professional.',
  'similarity_score': 0.643},
 {'category': 'Diabetes',
  'question': 'What is HbA1c?',
  'trusted_answer': 'HbA1c is a blood test that shows average blood sugar levels over about the past 2 to 3 months. It is often used to monitor diabetes control.',
  'similarity_score': 0.179}]

Beginner Explanation of Retrieval

If the user asks:

What is high blood pressure?

The retrieval function should find the dataset row about high blood pressure.

This means the chatbot is not answering only from the model.

It first retrieves trusted context from the dataset.

That is why this version is a real simple RAG system

Step 6: Load the Language Model

We will use TinyLlama.

TinyLlama is a small open-source chat model.

In Colab, go to:

Runtime
Change runtime type
T4 GPU

This will make the model run faster.

If you run on CPU, it may be slow.

In [8]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16
    )
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.to(device)

print("Model loaded successfully.")

Using device: cuda


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully.


What is float16
torch_dtype=torch.float16

This means:

use half precision
Why?

Reduces:

✅ GPU memory usage
✅ computation cost

Makes model faster.

Beginner Mental Model
float16 = compressed math for faster AI
CPU Version
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

No float16 used because:

CPU usually works better with normal precision.

Step 7: Create the Healthcare Safety System Prompt

The system prompt is the chatbot's rulebook.

It tells the model:

use simple language
answer for education only
do not diagnose
do not prescribe
encourage professional care
use the retrieved context

This is important in healthcare AI.

In [9]:
SYSTEM_PROMPT = """
You are a patient education assistant.

Rules:
1. Use simple, clear language.
2. Use the trusted context provided.
3. Do not diagnose the user.
4. Do not prescribe medication or treatment.
5. Do not claim certainty about the user's condition.
6. Encourage the user to contact a healthcare professional for personal medical advice.
7. For emergency symptoms, tell the user to seek urgent medical care.
8. Keep answers educational and supportive.
"""

Step 8: Create a RAG Prompt

This function prepares the message for the model.

It includes:

system rules
retrieved trusted context
user question

Pattern:

System instruction
↓
Trusted context
↓
User question
↓
Assistant answer

In [10]:
def format_retrieved_context(retrieved_items):
    context_text = ""

    for i, item in enumerate(retrieved_items, start=1):
        context_text += f"Context {i}\n"
        context_text += f"Category: {item['category']}\n"
        context_text += f"Trusted question: {item['question']}\n"
        context_text += f"Trusted answer: {item['trusted_answer']}\n\n"

    return context_text


def create_rag_prompt(user_question, retrieved_items):
    context_text = format_retrieved_context(retrieved_items)

    prompt = f"""
<|system|>
{SYSTEM_PROMPT}
</s>
<|user|>
Trusted context:
{context_text}

User question:
{user_question}

Answer the user using the trusted context. If the context is not enough, say that the user should ask a healthcare professional.
</s>
<|assistant|>
"""

    return prompt

In [11]:
sample_prompt = create_rag_prompt(
    user_question="How can I improve sleep quality?",
    retrieved_items=retrieve_context("How can I improve sleep quality?")
)

print(sample_prompt[:1500])


<|system|>

You are a patient education assistant.

Rules:
1. Use simple, clear language.
2. Use the trusted context provided.
3. Do not diagnose the user.
4. Do not prescribe medication or treatment.
5. Do not claim certainty about the user's condition.
6. Encourage the user to contact a healthcare professional for personal medical advice.
7. For emergency symptoms, tell the user to seek urgent medical care.
8. Keep answers educational and supportive.

</s>
<|user|>
Trusted context:
Context 1
Category: General Wellness
Trusted question: How can I improve my sleep quality?
Trusted answer: Good sleep habits include keeping a regular sleep schedule, reducing screen time before bed, limiting caffeine late in the day, creating a dark and quiet sleep environment, and relaxing before bedtime.

Context 2
Category: Mental Health
Trusted question: How can I manage stress?
Trusted answer: Stress management may include regular physical activity, sleep routines, relaxation breathing, talking with

Step 9: Generate a Model Response

This function sends the prompt to TinyLlama and returns the model answer.

Important settings:

max_new_tokens controls answer length
temperature controls randomness
lower temperature is safer and more predictable

For healthcare education, we keep temperature low.

In [16]:
def generate_model_response(prompt, max_new_tokens=220, temperature=0.3):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1800
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    response = response.strip()

    return response

Most Important Patterns You Should Learn
Pattern	Purpose
tokenizer()	text → tokens
model.generate()	AI generation
decode()	tokens → text
no_grad()	inference optimization
temperature	creativity control
max_new_tokens	response size
top_p	safe sampling
repetition_penalty	avoid looping

Step 10: Add Emergency Safety Check

Some user questions should not go to a normal chatbot response.

Example:

severe chest pain
trouble breathing
stroke symptoms
overdose
suicidal thoughts

For these, the chatbot should give an urgent safety message.

Pattern:

User question
↓
Check emergency keywords
↓
If emergency, return emergency message
↓
Else continue normal chatbot

In [12]:
emergency_keywords = [
    "chest pain",
    "can't breathe",
    "cannot breathe",
    "trouble breathing",
    "shortness of breath",
    "stroke",
    "face drooping",
    "arm weakness",
    "severe bleeding",
    "unconscious",
    "heart attack",
    "overdose",
    "poisoning",
    "suicide",
    "kill myself",
    "self harm",
    "severe allergic reaction",
    "anaphylaxis"
]


def check_emergency_keywords(user_question):
    question_lower = user_question.lower()

    for keyword in emergency_keywords:
        if keyword in question_lower:
            return True

    return False


def emergency_response():
    return (
        "This may be an emergency. Please seek urgent medical help now. "
        "If you are in immediate danger or have severe symptoms, call your local emergency number or go to the nearest emergency department. "
        "This chatbot cannot provide emergency medical care."
    )

Step 11: Add Output Safety Filter

Sometimes a model may say unsafe things like:

You have diabetes.
You should take this medication.
I diagnose you with asthma.

We want to catch those patterns.

If the answer looks unsafe, we replace it with a safer message.

Pattern:

Model answer
↓
Check unsafe phrases
↓
If unsafe, replace with safe message
↓
Else show answer

In [13]:
unsafe_phrases = [
    "you have",
    "you definitely have",
    "i diagnose",
    "you should take",
    "take this medication",
    "stop taking",
    "start taking",
    "you do not need a doctor",
    "ignore your doctor"
]


def response_has_unsafe_claim(response):
    response_lower = response.lower()

    for phrase in unsafe_phrases:
        if phrase in response_lower:
            return True

    return False


def safe_replacement_response():
    return (
        "I can provide general health education, but I cannot diagnose you or tell you what treatment to take. "
        "Please speak with a qualified healthcare professional for personal medical advice."
    )

Step 12: Build the Final RAG Chatbot Function

This is the main function.

It combines:

emergency check
retrieval
prompt creation
model generation
output safety filter
disclaimer

This is the full safe chatbot pipeline.

In [14]:
def patient_education_chatbot(user_question, top_k=2):
    if check_emergency_keywords(user_question):
        return {
            "question": user_question,
            "retrieved_context": [],
            "answer": emergency_response(),
            "safety_status": "emergency_redirect"
        }

    retrieved_items = retrieve_context(user_question, top_k=top_k)

    prompt = create_rag_prompt(user_question, retrieved_items)

    model_answer = generate_model_response(prompt)

    if response_has_unsafe_claim(model_answer):
        final_answer = safe_replacement_response()
        safety_status = "unsafe_output_replaced"
    else:
        final_answer = model_answer
        safety_status = "safe_generated_answer"

    disclaimer = (
        "\n\nEducational note: This information is for general education only and does not replace advice from a healthcare professional."
    )

    final_answer = final_answer + disclaimer

    return {
        "question": user_question,
        "retrieved_context": retrieved_items,
        "answer": final_answer,
        "safety_status": safety_status
    }

Step 13: Test the Chatbot

Now we test the chatbot with a normal patient education question.

We will print:

user question
retrieved dataset context
chatbot answer
safety status

In [17]:
result = patient_education_chatbot("What is high blood pressure and why is it dangerous?")

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED CONTEXT:")
for item in result["retrieved_context"]:
    print("-", item["question"], "| similarity:", item["similarity_score"])

print("\nANSWER:")
print(result["answer"])

print("\nSAFETY STATUS:")
print(result["safety_status"])

QUESTION:
What is high blood pressure and why is it dangerous?

RETRIEVED CONTEXT:
- What is high blood pressure and why is it dangerous? | similarity: 0.624
- What is HbA1c? | similarity: 0.15

ANSWER:
Sure! Here's an updated version with the additional information on high blood pressure and its potential consequences:

Trusted Context:
Cardiology

Question: What is high blood pressure and why is it dangerous?
Answer: High blood pressure (also known as hypertension) is the force of blood against artery walls that is consistently too high. This can lead to several health problems, including heart disease, stroke, kidney disease, and other complications. High blood pressure can be caused by various factors such as genetics, lifestyle choices, and environmental factors. However, lifestyle changes and medications may help manage this condition.

Trusted Context:
Diabetes

Question: What is HbA1c?
Answer: HbA1c is a blood test that measures average blood sugar levels over about the past 2 

Step 14: Test Another Question

This test checks whether the system retrieves sleep-related context.

In [18]:
result = patient_education_chatbot("How can I sleep better at night?")

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED CONTEXT:")
for item in result["retrieved_context"]:
    print("-", item["question"], "| similarity:", item["similarity_score"])

print("\nANSWER:")
print(result["answer"])

print("\nSAFETY STATUS:")
print(result["safety_status"])

QUESTION:
How can I sleep better at night?

RETRIEVED CONTEXT:
- How can I improve my sleep quality? | similarity: 0.602
- How can I manage stress? | similarity: 0.166

ANSWER:
Trusted Context: General Wellness
Question: How can I improve my sleep quality?
Answer: Good sleep habits include keeping a regular sleep schedule, reducing screen time before bed, limiting caffeine late in the day, creating a dark and quiet sleep environment, and relaxing before bedtime.

Trusted Answer:
- Reducing screen time before bed: Turn off your phone, computer, or TV at least an hour before going to bed.
- Limiting caffeine late in the day: Avoid drinking coffee or tea within two hours of bedtime.
- Creating a dark and quiet sleep environment: Make sure your room is cool, dark, and quiet.
- Relaxing before bedtime: Take a warm bath, read a book, or listen to calming music.

If the user still has questions, they should consult their healthcare provider for further guidance on how to improve their sleep q

Step 15: Test Emergency Safety

This question should not generate a normal educational answer.

It should trigger the emergency safety response.

In [19]:
result = patient_education_chatbot("I am having severe chest pain and trouble breathing.")

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSAFETY STATUS:")
print(result["safety_status"])

QUESTION:
I am having severe chest pain and trouble breathing.

ANSWER:
This may be an emergency. Please seek urgent medical help now. If you are in immediate danger or have severe symptoms, call your local emergency number or go to the nearest emergency department. This chatbot cannot provide emergency medical care.

SAFETY STATUS:
emergency_redirect


Step 16: Create a Simple Interactive Chatbot

This lets you ask questions repeatedly inside Colab.

To stop, type:

quit

In [20]:
def interactive_chatbot():
    print("Patient Education Chatbot")
    print("Type 'quit' to stop.")
    print("-" * 60)

    while True:
        user_question = input("\nYour question: ")

        if user_question.lower().strip() in ["quit", "exit", "bye"]:
            print("Goodbye.")
            break

        result = patient_education_chatbot(user_question)

        print("\nAnswer:")
        print(result["answer"])

        print("\nSafety status:", result["safety_status"])
        print("-" * 60)


# Uncomment the line below to start the chatbot:
# interactive_chatbot()

tep 17: Evaluate the Chatbot on Sample Questions

This is a simple evaluation.

We will test the chatbot on a few questions from the dataset.

We will collect:

question
retrieved context
answer length
safety status

This does not prove medical correctness.

It helps us inspect chatbot behavior.

In [21]:
test_questions = [
    "What is HbA1c?",
    "How can I lower cholesterol?",
    "What are signs of anxiety?",
    "What are symptoms of pneumonia?",
    "Why should I tell my doctor about medications?"
]

evaluation_rows = []

for question in test_questions:
    result = patient_education_chatbot(question)

    retrieved_questions = [
        item["question"] for item in result["retrieved_context"]
    ]

    evaluation_rows.append({
        "question": question,
        "retrieved_questions": retrieved_questions,
        "answer": result["answer"],
        "answer_word_count": len(result["answer"].split()),
        "safety_status": result["safety_status"]
    })

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df

,question,retrieved_questions,answer,answer_word_count,safety_status
0,What is HbA1c?,"[What is HbA1c?, What should I do if I miss a ...",Sure! Here's an updated version of the rules w...,146,safe_generated_answer
1,How can I lower cholesterol?,"[How can I lower cholesterol naturally?, What ...","Rules:\n1. Use simple, clear language.\n2. Use...",134,safe_generated_answer
2,What are signs of anxiety?,"[What are common signs of anxiety?, What are e...",Sure! Here's an updated version of the rules w...,148,safe_generated_answer
3,What are symptoms of pneumonia?,"[What are symptoms of pneumonia?, What are com...",Trusted Context:\nCategory: Respiratory Health...,86,safe_generated_answer
4,Why should I tell my doctor about medications?,[Why is it important to tell doctors about all...,Sure! Here's an updated version with additiona...,154,safe_generated_answer


In [22]:
print("Average answer word count:", round(evaluation_df["answer_word_count"].mean(), 1))
print("Shortest answer:", evaluation_df["answer_word_count"].min())
print("Longest answer:", evaluation_df["answer_word_count"].max())

evaluation_df["safety_status"].value_counts()

Average answer word count: 133.6
Shortest answer: 86
Longest answer: 154


,count
safety_status,
safe_generated_answer,5


Step 18: Compare RAG Context Retrieval

This table shows which dataset rows were retrieved for each test question.

This helps us check whether the retrieval system is working.

In [23]:
for i in range(len(evaluation_df)):
    print("QUESTION:", evaluation_df.iloc[i]["question"])
    print("RETRIEVED:")
    for retrieved_question in evaluation_df.iloc[i]["retrieved_questions"]:
        print("-", retrieved_question)
    print("-" * 80)

QUESTION: What is HbA1c?
RETRIEVED:
- What is HbA1c?
- What should I do if I miss a medication dose?
--------------------------------------------------------------------------------
QUESTION: How can I lower cholesterol?
RETRIEVED:
- How can I lower cholesterol naturally?
- What should I do if I miss a medication dose?
--------------------------------------------------------------------------------
QUESTION: What are signs of anxiety?
RETRIEVED:
- What are common signs of anxiety?
- What are early signs of diabetes?
--------------------------------------------------------------------------------
QUESTION: What are symptoms of pneumonia?
RETRIEVED:
- What are symptoms of pneumonia?
- What are common symptoms of a heart attack?
--------------------------------------------------------------------------------
QUESTION: Why should I tell my doctor about medications?
RETRIEVED:
- Why is it important to tell doctors about all my medications?
- What is high blood pressure and why is it dangero

Step 19: Save Project Outputs

We save:

the medical Q&A dataset
the chatbot evaluation results

These files are useful for GitHub.

In [24]:
medical_df.to_csv("medical_patient_education_qa_dataset.csv", index=False)
evaluation_df.to_csv("chatbot_evaluation_results.csv", index=False)

print("Saved files:")
print("medical_patient_education_qa_dataset.csv")
print("chatbot_evaluation_results.csv")

Saved files:
medical_patient_education_qa_dataset.csv
chatbot_evaluation_results.csv
